# LIBERO-plus 相机实验：运行与抓取诊断

更新：2026-10-31。已完成三组单回合试运行与首次抓取的视频标注。保留历史单元及输出。
本轮只准备默认相机与 view_15 的配对重复，采集代码由助手负责；相对位置计算留给用户。

**当前无需连接 GPU 完成的工作：**阅读 practice_notes.md 的 I 节，完成文中的三个短问题。
之后亲手补第 15 节的 relative_position()；现在没有真实位置日志，不编造数值或运行结果。

**重新连接后的启动顺序（不要“全部运行”）：**

| 会话状态 | 执行顺序 |
| --- | --- |
| 原会话仍在且依赖/资源完整 | 第 1、4、5 节 → 10a 恢复/检查参考审计 → 11（含采集定义与评测函数）→ 14 |
| 全新会话 | 第 1、2、3 节 → 重启内核 → 第 1、4、5、6 节 → 10a → 11 → 14 |

跳过已完成的 7–9 节场景审计和 12–13 节旧试运行。第 14 节先做不加载策略的采集接口检查，
通过后才运行默认条件和 view_15。第一对确认产物后，再运行额外两对种子；每回合结束自动归档。
第 15 节是你的分析练习，尚未填写时抛出 NotImplementedError 是预期行为。

本地终端仍是 CPU；新增接口仅通过本地替身检查，**真实 Plus 接口、更新后的安装及 GPU 采集待验证**。
安装依据：[LeRobot v0.6.1 Dockerfile](https://github.com/huggingface/lerobot/blob/v0.6.1/docker/Dockerfile.benchmark.libero_plus)。

**当前新步骤：**第 16 节先补 first_true_time() 并生成六行真实汇总表；第 17 节为执行窗口 50/10 对照框架。
新 GPU 调用默认注释。第 15 节用户实现与全部历史结果保留。


## 1．确认会话与工作目录

这里使用专门的源码目录，和原来的 `/content/lerobot` 分开。
本节打印磁盘空间，供后续下载扩展资源前判断；此时不下载资源。

In [ ]:
import sys
import shutil
from pathlib import Path

LEROBOT_REPO = Path("/content/lerobot-camera-study")
PLUS_REPO = Path("/content/LIBERO-plus-camera-study")
PLUS_SHA = "4976dc30028e805ff8094b55501d532c48fec182"

print("Python:", sys.executable, sys.version)
print("可用磁盘 GiB:", round(shutil.disk_usage("/content").free / 1024**3, 2))
assert sys.version_info >= (3, 12), "LeRobot 0.6.1 需要 Python >= 3.12"

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
# 记录 Notebook 通信/显示组件，便于区分模型问题与播放器问题。
from importlib.metadata import version, PackageNotFoundError
for package in ["IPython", "jupyter-client", "transformers", "tokenizers"]:
    try:
        print(package + ":", version(package))
    except PackageNotFoundError:
        print(package + ": 尚未安装")


## 2．取得固定版本源码

LeRobot 使用 `v0.6.1`；LIBERO-plus 固定到已经核对任务 ID 的 commit。
`--no-checkout` 先不展开默认分支，再取指定版本，避免把最新分支当作已核对版本。
这是新会话的专用目录，不在本单元操作原来的项目仓库。

In [ ]:
import subprocess

for repository_url, directory, revision in [
    ("https://github.com/huggingface/lerobot.git", LEROBOT_REPO, "v0.6.1"),
    ("https://github.com/sylvestf/LIBERO-plus.git", PLUS_REPO, PLUS_SHA),
]:
    if not directory.exists():
        subprocess.run(
            ["git", "clone", "--filter=blob:none", "--no-checkout",
             repository_url, str(directory)], check=True,
        )
    subprocess.run(
        ["git", "-C", str(directory), "fetch", "--depth=1", "origin", revision],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(directory), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
    subprocess.run(["git", "-C", str(directory), "rev-parse", "HEAD"], check=True)

## 3．直接安装 LIBERO-plus 所需依赖，完成后重启内核

2026-10-30 第二次修正：先前绕经普通 `hf-libero` 的方案会引入 `robomimic → egl_probe`
和 `hf-libero → hf-egl-probe`，当前 Colab 在构建这两个辅助包的 wheel 时失败。
原日志没有提供底层编译错误，因此不能把它直接归因于 Python 或 CMake 的某个版本。

现在直接安装 LeRobot 的 dataset / smolvla / evaluation / scipy extras，
再在**同一次依赖求解**中指定 Plus 的仿真运行依赖，不选择普通 `[libero]` extra。
移除已有的 `hf-libero`，使用固定版本的 Plus 源码提供 `libero` 导入入口。
MuJoCo 仍使用 `MUJOCO_GL=egl` 渲染；实际图形与场景加载留待后续验证。

此依赖路径依据 Plus 的环境源码、额外依赖文件以及 robosuite 1.4.1 的依赖元数据核对。
这一步只面向 LeRobot 的仿真评测，不安装 Plus 自带的 lifelong 训练框架。

子进程输出同步显示并写入独立日志目录；pip 还会生成机器可读安装报告。
当前以本地 VS Code 文件为代码主版本，由用户通过 Colab 插件执行；助手的终端仍是本地 CPU。

新建会话时运行本节，安装成功后重启内核，再执行 1 → 4 → 5。已有成功会话不要为了视频展示问题重装。

本次保存的输出已确认安装、真实渲染及闭环评测可运行。后续环境固定已跑通的 Transformers / Tokenizers，
并使用 `jupyter-client==8.10.0` 避免旧通信组件的 `datetime.utcnow()` 弃用提示。
该版本仍属 8.x，使用时区明确的 UTC 时间；[官方版本元数据](https://pypi.org/project/jupyter-client/8.10.0/)。
预装 `dopamine-rl` 与本项目的 Gym 要求冲突，在这个专用会话中移除；不降级项目需要的 Gym。
**现有会话无需为了看视频重装或重启**。安装单元的更新供下次配置使用，安装输出保留的是修改前的历史记录。


In [ ]:
import subprocess
import time

log_dir = Path("/content/libero-plus-install-logs") / str(time.time_ns())
log_dir.mkdir(parents=True, exist_ok=True)


def run_logged(command, step_name):
    log_path = log_dir / f"{step_name}.log"
    print(f"开始 {step_name}；完整日志：{log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding="utf-8")[-6000:]
        raise RuntimeError(
            f"{step_name} 失败，退出码 {returncode}。完整日志：{log_path}\n{tail}"
        )


run_logged(["apt-get", "update", "-qq"], "01-system-update")
run_logged(
    ["apt-get", "install", "-y", "libexpat1", "libfontconfig1-dev", "libmagickwand-dev", "unzip"],
    "02-system-packages",
)

# 本 Notebook 使用独立的机器人评测会话，不使用 Colab 预装的 Dopamine。
# Dopamine 4.1.2 要求 gym<=0.25.2，与 Plus 所需 gym==0.26.2 冲突；保留项目的 Gym。
from importlib.metadata import version, PackageNotFoundError
remove_packages = []
for package in ["hf-libero", "dopamine-rl"]:
    try:
        print("移除当前实验不使用的包：", package, version(package))
        remove_packages.append(package)
    except PackageNotFoundError:
        pass
if remove_packages:
    run_logged(
        [sys.executable, "-m", "pip", "uninstall", "-y", *remove_packages],
        "03-remove-conflicting-packages",
    )

# 在同一次求解中检查 LeRobot 和 Plus 的全部选中依赖。
# 不选择 [libero] extra，从源头避免 hf-libero / robomimic / EGL probe 的依赖分支。
run_logged(
    [sys.executable, "-m", "pip", "install", "--report", str(log_dir / "pip-report.json"),
     "-e", f"{LEROBOT_REPO}[dataset,smolvla,evaluation,scipy-dep]",
     "robosuite==1.4.1", "bddl==1.0.1", "easydict==1.13", "mujoco==3.7.0",
     "matplotlib==3.10.8", "Wand==0.6.13", "scikit-image==0.25.2", "gym==0.26.2",
     "hydra-core>=1.2,<1.4", "cloudpickle>=2", "h5py>=3.11,<4",
     "transformers==5.5.4", "tokenizers==0.22.2", "jupyter-client==8.10.0"],
    "04-lerobot-plus-runtime",
)

# Plus 的 setup.py 没有声明完整运行依赖，上一步已显式安装。
run_logged(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PLUS_REPO)],
    "05-plus-editable",
)
print("安装步骤结束。完整日志目录：", log_dir)
print("现在重启内核，然后执行 1 → 4 → 5。")


## 4．设置图形后端、源码路径与独立配置

在导入 MuJoCo、robosuite 或 LIBERO 前执行。
`sys.path` 管当前内核，`PYTHONPATH` 供以后启动的子进程使用。
独立 `LIBERO_CONFIG_PATH` 避免导入时再次询问路径，也避免沿用普通 LIBERO 的旧配置。
目前只写资源路径；资源目录是否真正完整，要在下一阶段验证。

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"
os.environ["LIBERO_CONFIG_PATH"] = "/content/libero-plus-camera-config"

source_roots = [str(LEROBOT_REPO / "src"), str(PLUS_REPO)]
for source_root in reversed(source_roots):
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
os.environ["PYTHONPATH"] = os.pathsep.join(source_roots)

package_root = PLUS_REPO / "libero" / "libero"
assert (package_root / "benchmark" / "libero_suite_task_map.py").is_file()
config_dir = Path(os.environ["LIBERO_CONFIG_PATH"])
config_dir.mkdir(parents=True, exist_ok=True)
config = {
    "benchmark_root": str(package_root),
    "bddl_files": str(package_root / "bddl_files"),
    "init_states": str(package_root / "init_files"),
    "datasets": str(PLUS_REPO / "libero" / "datasets"),
    "assets": str(package_root / "assets"),
}

import yaml
(config_dir / "config.yaml").write_text(yaml.safe_dump(config))
print((config_dir / "config.yaml").read_text())
# 按 robosuite 官方 setup_macros 的方式建立默认私有配置；保留已有自定义文件。
# find_spec 只定位顶层包，避免先导入 robosuite 才触发“未配置”提示。
import importlib.util
suite_spec = importlib.util.find_spec("robosuite")
if suite_spec is None or suite_spec.origin is None:
    raise ImportError("未找到 robosuite，请先完成安装单元")
suite_dir = Path(suite_spec.origin).parent
private_macros = suite_dir / "macros_private.py"
if not private_macros.exists():
    shutil.copyfile(suite_dir / "macros.py", private_macros)
    print("已建立 robosuite 默认私有配置：", private_macros)


## 5．检查实际导入来源与候选任务

检查实际子模块的位置，而非仅检查安装版本号。Plus 的顶层 `libero` 可能是命名空间包，
所以不能单凭顶层 `__file__=None` 判定这次导入失败。
本节成功只说明导入和任务元数据可读取，不代表渲染或完整评测成功。

In [ ]:
import inspect
from importlib.metadata import version

import lerobot
from libero.libero import benchmark

print("LeRobot 版本:", version("lerobot"))
print("LeRobot 实际源码:", lerobot.__file__)
print("LIBERO benchmark 实际源码:", inspect.getfile(benchmark))
assert Path(inspect.getfile(benchmark)).resolve().is_relative_to(PLUS_REPO.resolve())

object_tasks = benchmark.libero_task_map["libero_object"]
print("Object 条目数:", len(object_tasks))
assert len(object_tasks) == 2518
for task_id in [0, 653, 654]:
    print(task_id, object_tasks[task_id])

## 环境准备已通过，继续执行第 6 节

2026-10-30 云端已保存的运行输出确认：LeRobot 0.6.1、正确的 Plus 源码路径、2518 个 Object 条目。
已执行的单元及其输出保留；下面新增资源准备与真实环境记录。
接下来按 6 → 7 → 8 执行，不必重跑安装或重新启动会话。
新增单元只做静态和小样本辅助检查，实际资源下载与场景加载尚待 Colab 执行。


## 6．准备固定版本的扩展资源

官方 `assets.zip` 的实际大小为 **6,395,849,578 字节，约 5.96 GiB**，还需要解压空间。
本节会使用当前 Colab 的磁盘与网络，不加载策略权重；下载和解压可能需要几分钟。
先读运行输出中的可用空间，再执行下载单元。

来源：[官方资源数据集](https://huggingface.co/datasets/Sylvest/LIBERO-plus/tree/dd2bd61b7d9a6fef1abc52d606e983b41886a149)。
源码中的相机设置与资源路径依据 [LeRobot 规范配置](https://github.com/huggingface/lerobot/blob/v0.6.1/docker/Dockerfile.benchmark.libero_plus)。
压缩包内部可能有作者目录前缀，因此按实际 ZIP 路径找到 `assets`，直接解压到包所需位置，不额外保存第二份解压目录。

In [ ]:
import json
import shutil

ASSET_REVISION = "dd2bd61b7d9a6fef1abc52d606e983b41886a149"
ASSET_ZIP_BYTES = 6395849578
asset_dir = package_root / "assets"
asset_marker = asset_dir / ".lerobot_camera_assets.json"
assets_ready = (
    asset_marker.is_file()
    and json.loads(asset_marker.read_text()).get("revision") == ASSET_REVISION
    and (asset_dir / "scenes").is_dir()
    and (asset_dir / "textures").is_dir()
)
free_bytes = shutil.disk_usage("/content").free
print("当前可用磁盘 GiB:", round(free_bytes / 1024**3, 2))
print("资源 ZIP GiB:", round(ASSET_ZIP_BYTES / 1024**3, 2))
print("固定版本扩展资源已准备:", assets_ready)
print("最终资源位置:", asset_dir)

In [ ]:
from pathlib import PurePosixPath
import zipfile
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

if assets_ready:
    print("已存在完整解压标记，跳过资源下载与解压。")
else:
    if free_bytes < ASSET_ZIP_BYTES + 2 * 1024**3:
        raise RuntimeError("当前空间不足以保存资源包并预留空间，请先检查磁盘。")
    zip_path = Path(hf_hub_download(
        repo_id="Sylvest/LIBERO-plus", repo_type="dataset", filename="assets.zip",
        revision=ASSET_REVISION, local_dir="/content/libero-plus-download", token=False,
    ))
    assert zip_path.stat().st_size == ASSET_ZIP_BYTES, "资源包大小与固定版本元数据不符"

    with zipfile.ZipFile(zip_path) as archive:
        entries = []
        roots = set()
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts
            if "assets" not in parts or member.is_dir():
                continue
            index = parts.index("assets")
            roots.add(parts[:index + 1])
            relative = parts[index + 1:]
            if not relative or ".." in relative:
                raise RuntimeError(f"资源包路径异常：{member.filename}")
            entries.append((member, relative))
        if len(roots) != 1 or not entries:
            raise RuntimeError(f"需要人工检查 ZIP 中的 assets 根目录：{roots}")

        unpacked_bytes = sum(member.file_size for member, _ in entries)
        free_after_download = shutil.disk_usage("/content").free
        print("解压数据 GiB:", round(unpacked_bytes / 1024**3, 2))
        print("下载后剩余 GiB:", round(free_after_download / 1024**3, 2))
        if free_after_download < unpacked_bytes + 2 * 1024**3:
            raise RuntimeError("下载已完成，但解压空间不足；保留 ZIP，先处理磁盘空间。")

        for member, relative in tqdm(entries, desc="解压 Plus assets", unit="file"):
            target = asset_dir.joinpath(*relative)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)

    assert (asset_dir / "scenes").is_dir()
    assert (asset_dir / "textures").is_dir()
    asset_marker.write_text(json.dumps({
        "revision": ASSET_REVISION, "zip_bytes": ASSET_ZIP_BYTES,
        "files": len(entries), "unpacked_bytes": unpacked_bytes,
    }, indent=2))
    print("资源准备完成：", asset_dir)

## 7．通过 LeRobot 实际加载两个候选场景

两组都使用第 0 条初始化数组、seed=1000、20 Hz、相对控制、10 个稳定等待步与 256×256 图像。
这只是相同配置，是否得到相同物理状态，需要用下面保存的快照核对。
每组保存两路图像，以及运行时相机位置/朝向、FOV、正式 BDDL 语言与实际任务描述。

原始 MuJoCo 图像按官方 `LiberoProcessorStep` 的方式旋转 180° 后保存，便于对应策略看到的方向。
此时尚未运行完整策略预处理器或生成动作。
`task.language` 和基础 BDDL 的语言分别记录，不预先隐藏两者的差异。

输出放在 `/content/outputs/camera_pair_audit_001`。成功后重做新实验请换目录名；失败时保留已有文件帮助排错。

In [ ]:
import hashlib
import io
import random
from contextlib import redirect_stdout
import numpy as np
from PIL import Image as PILImage
from lerobot.envs.libero import LiberoEnv

AUDIT_DIR = Path("/content/outputs/camera_pair_audit_001")
assert not (AUDIT_DIR / "audit.json").exists(), "该实验已有完整记录，请换一个 AUDIT_DIR"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# 官方构造函数会打印完整任务顺序，保存到日志中，避免长列表淹没主要输出。
suite_log = io.StringIO()
with redirect_stdout(suite_log):
    suite = benchmark.get_benchmark_dict()["libero_object"]()
(AUDIT_DIR / "suite_creation.log").write_text(suite_log.getvalue())

records = []
sim_states = {}
for task_id in [653, 654]:
    random.seed(1000)
    np.random.seed(1000)
    task = suite.get_task(task_id)
    base_name = task.name.split("_view_", 1)[0]
    bddl_path = package_root / "bddl_files" / "libero_object" / f"{base_name}.bddl"
    init_path = package_root / "init_files" / "libero_object" / f"{base_name}.pruned_init"
    env = None
    try:
        env = LiberoEnv(
            task_suite=suite, task_id=task_id, task_suite_name="libero_object",
            observation_height=256, observation_width=256, obs_type="pixels_agent_pos",
            camera_name_mapping={"agentview_image": "camera1", "robot0_eye_in_hand_image": "camera2"},
            init_states=True, episode_index=0, n_envs=1, num_steps_wait=10,
            control_freq=20, control_mode="relative", is_libero_plus=True, hard_reset=True,
        )
        init_index = env.init_state_id % len(env._init_states)
        selected_init = np.asarray(env._init_states[init_index]).copy()
        observation, info = env.reset(seed=1000)
        sim = env._env.sim
        raw_data = getattr(sim.data, "_data", sim.data)

        # 仿真数组是可变缓冲区，关闭/推进环境之前复制成独立快照。
        sim_states[task_id] = {
            name: torch.from_numpy(np.array(getattr(raw_data, name), copy=True))
            for name in ["qpos", "qvel", "act", "mocap_pos", "mocap_quat"]
        }
        sim_states[task_id]["selected_init"] = torch.from_numpy(selected_init)
        cameras = {}
        for name in ["agentview", "robot0_eye_in_hand"]:
            camera_id = sim.model.camera_name2id(name)
            cameras[name] = {
                "model_position": np.asarray(sim.model.cam_pos[camera_id]).tolist(),
                "model_quaternion": np.asarray(sim.model.cam_quat[camera_id]).tolist(),
                "world_position": np.asarray(raw_data.cam_xpos[camera_id]).tolist(),
                "world_rotation_matrix": np.asarray(raw_data.cam_xmat[camera_id]).reshape(3, 3).tolist(),
                "fovy_degrees": float(sim.model.cam_fovy[camera_id]),
            }
        for key, image in observation["pixels"].items():
            PILImage.fromarray(image[::-1, ::-1].copy()).save(AUDIT_DIR / f"task_{task_id}_{key}.png")
        record = {
            "task_id": task_id, "task_name": task.name, "base_task": base_name,
            "released_language": env.task_description,
            "base_bddl_language": env._env.language_instruction,
            "base_bddl": str(bddl_path), "base_bddl_sha256": hashlib.sha256(bddl_path.read_bytes()).hexdigest(),
            "init_state_file": str(init_path), "init_state_index": int(init_index),
            "seed": 1000, "settling_steps": 10, "simulation_time": float(raw_data.time),
            "robot": type(env._env.robots[0].robot_model).__name__,
            "control_mode": env.control_mode, "control_frequency": env.control_freq,
            "initial_success": bool(env._env.check_success()), "cameras": cameras,
        }
        records.append(record)
        print("已记录：", task_id, "qpos shape:", tuple(sim_states[task_id]["qpos"].shape))
        print("任务描述：", record["released_language"])
        print("基础 BDDL 语言：", record["base_bddl_language"])
        print("外部相机实际位置：", cameras["agentview"]["world_position"])
    except Exception as error:
        (AUDIT_DIR / "audit_partial.json").write_text(json.dumps({
            "completed_tasks": records, "failed_task_id": task_id,
            "error": f"{type(error).__name__}: {error}",
        }, ensure_ascii=False, indent=2))
        raise
    finally:
        if env is not None:
            env.close()

torch.save(sim_states, AUDIT_DIR / "sim_states.pt")
(AUDIT_DIR / "audit.json").write_text(json.dumps({
    "status": "snapshots_collected_not_yet_compared",
    "libero_plus_revision": PLUS_SHA, "asset_revision": ASSET_REVISION,
    "resolution": [256, 256], "tasks": records,
}, ensure_ascii=False, indent=2))
print("快照采集完成，尚未判定两组条件是否只差相机：", AUDIT_DIR)

## 8．看真实图像，再由你检查一项状态差异

图中上、下两行分别是任务 653 / 654，左、右两列分别是外部相机与腕部相机。
先观察：外部相机改变了什么，腕部图像是否也变化，以及打印的两种语言有何差异。

然后使用已经熟悉的 Tensor 操作，自行补一行：**两组 `qpos` 之差的绝对值最大值是多少？**
可以沿用 `.abs()`、`.max()`、`.item()`；不需要再写一套指标框架。
这里是未标准化的仿真状态。即便 qpos 一致，也还需要检查 qvel、初始化索引及其他字段，不能直接宣称全部条件相同。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, task_id in enumerate([653, 654]):
    for column, camera in enumerate(["camera1", "camera2"]):
        axes[row, column].imshow(PILImage.open(AUDIT_DIR / f"task_{task_id}_{camera}.png"))
        axes[row, column].set_title(f"task {task_id} / {camera}")
        axes[row, column].axis("off")
fig.tight_layout()
fig.savefig(AUDIT_DIR / "camera_pair.png", dpi=150)
plt.show()

qpos_a = sim_states[653]["qpos"]
qpos_b = sim_states[654]["qpos"]
print("qpos shapes:", qpos_a.shape, qpos_b.shape)
# 你补一行：作差 → 绝对值 → 最大值 → Python 数字，并打印结果。
print("qpos difference (max):", (qpos_a - qpos_b).abs().max().item())

In [ ]:
for key in sim_states[653].keys():
    if key not in sim_states[654]:
        print(f"task 653 has {key}, but task 654 does not.")
    else:
        shape_a = sim_states[653][key].shape
        shape_b = sim_states[654][key].shape
        if shape_a != shape_b:
            print(f"task 653 and 654 have different shapes for {key}: {shape_a} vs {shape_b}")
        else:
            print(f"{key} shape is {shape_a}")
            if not torch.allclose(sim_states[653][key], sim_states[654][key]):
                print(f"{key} value difference (max):", (sim_states[653][key] - sim_states[654][key]).abs().max().item())
# print(sim_states[653].keys())

## 9．保存本次小型核对结果到云盘（采集成功后执行）

只复制图像、JSON 和状态快照，不复制 6 GB 的资源包。
资源下载与代码准备是工程接入；相机与状态是否满足对照要求，仍需根据实际记录解释。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/robot-lab/experiments") / AUDIT_DIR.name
assert (AUDIT_DIR / "audit.json").is_file(), "先完成场景快照采集"
assert not destination.exists(), "云盘已有同名结果，请为新实验换目录名"
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(AUDIT_DIR, destination)
print("已保存：", destination)

## 10．已有核对结果：状态相同，相机与任务描述有差异

保留上面用户编写的 qpos 最大差与全字段检查。
当前 `sim_states` 只包含物理状态，不包含相机参数和图像；它们分别保存在 `records` / `audit.json` 与 PNG 中。
已从本次保存的实际数据核对：qpos、qvel、selected_init 逐元素相同；act 与 mocap 数组为空。
外部相机实际位置分别为 `[0.9048,0.0158,0.8872]` 和 `[0.9043,0.0316,0.8872]`，位移约 1.58 cm，FOV 均为 45°。
腕部相机参数和 PNG 文件相同。基础 BDDL 语言相同，但发布版任务描述分别带 view 1 / view 2，因此语言也被改变了。

`torch.allclose` 检查容差内接近，`torch.equal` 检查形状与元素值相同；空数组没有可比较的物理数值。
这些数据支持已记录起始条件的核对，不证明两个 MuJoCo 模型的所有字段完全一致。

## 10a．恢复已有参考审计，避免重新执行第 7–9 节

先挂载 Drive，读取已归档的 camera_pair_audit_001。断线不一定等于运行时删除，已有本地目录就核对内容再使用。
如果 VS Code 中 Drive 授权未完成，请在 Colab 网页完成这一次挂载后返回；不需要因此重装环境。


In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import numpy as np
import torch
from lerobot.envs.libero import LiberoEnv

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

AUDIT_DIR = Path("/content/outputs/camera_pair_audit_001")
reference_archive = Path("/content/drive/MyDrive/robot-lab/experiments/camera_pair_audit_001")
assert reference_archive.is_dir(), "未找到已归档参考审计，请核对 Drive 路径"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
for filename in ["audit.json", "sim_states.pt"]:
    source = reference_archive / filename
    target = AUDIT_DIR / filename
    assert source.is_file(), f"参考审计缺少 {filename}"
    if target.exists():
        assert source.read_bytes() == target.read_bytes(), f"本地与归档的 {filename} 不同，停止覆盖"
    else:
        shutil.copy2(source, target)
reference_report = json.loads((AUDIT_DIR / "audit.json").read_text())
assert reference_report["libero_plus_revision"] == PLUS_SHA
ASSET_REVISION = reference_report["asset_revision"]
marker = package_root / "assets/.lerobot_camera_assets.json"
assert marker.is_file() and json.loads(marker.read_text())["revision"] == ASSET_REVISION, "资源标记不符，请执行第 6 节"
assert all((package_root / "assets" / name).is_dir() for name in ["scenes", "textures"]), "资源未准备，请执行第 6 节"
base_bddl = package_root / "bddl_files/libero_object" / (reference_report["tasks"][0]["base_task"] + ".bddl")
assert hashlib.sha256(base_bddl.read_bytes()).hexdigest() == reference_report["tasks"][0]["base_bddl_sha256"]
print("参考审计及基础 BDDL 已核对：", AUDIT_DIR)


## 11．下一里程碑：一次可复现的闭环基线

现在开始模型真正执行任务：观察 → 官方预处理 → SmolVLA 动作队列 → 官方后处理 → 环境 step → 新观察。
复用 LeRobot 的完整评测入口，不另写 rollout；新增部分只负责固定实验条件与记录起始状态。

三个条件预先定义为：默认相机、ID 653 的视角、ID 666 的更大视角。666 在固定任务表中为 `view_15_15_100_0_0_initstate_0`。
这轮先运行默认相机的一回合，查看视频后再运行其余条件。单回合只是排错试运行，不能估计可靠成功率。

默认条件从同一个基础 BDDL 创建，并不是将 ID 653 误称为默认相机：临时替换该列表槽位的 Task 描述。
三组都使用基础 BDDL 的正式语言；临时变更在函数结束或异常时恢复。
因此这属于本项目的受控实验，不宣称逐字复现发布版 benchmark 的语言协议。

会话仍在时，直接从本节开始，不必重跑安装、解压和场景采集。
下面会下载策略 checkpoint，首次加载策略还可能下载 VLM 权重；这是接下来需要等待的主要工程步骤。

In [ ]:
from huggingface_hub import snapshot_download

pilot_audit = json.loads((AUDIT_DIR / "audit.json").read_text())
pilot_reference = torch.load(AUDIT_DIR / "sim_states.pt", map_location="cpu", weights_only=True)[653]
pilot_base_task = pilot_audit["tasks"][0]["base_task"]
pilot_instruction = pilot_audit["tasks"][0]["base_bddl_language"]
PILOT_ROOT = Path("/content/outputs/smolvla_camera_pilot_001")
CHECKPOINT = "lerobot/smolvla_libero"

# 固定此次实际获取的模型版本，后续条件共用同一个本地快照。
# 这是公开模型，显式匿名读取，避免 VS Code 会话尝试访问 Colab UI 的密钥接口。
checkpoint_revision = "a24f1f380461b665158d52acd140152c79e8b36d"  # 本次已完成试运行的版本
checkpoint_snapshot = Path(snapshot_download(
    repo_id=CHECKPOINT, revision=checkpoint_revision,
    allow_patterns=["*.json", "*.safetensors"], token=False,
))
print("策略版本：", checkpoint_revision)
print("本地快照：", checkpoint_snapshot)
print("三个条件共同使用的任务指令：", pilot_instruction)

### 评测接入与起始条件检查

这一单元由助手准备，用户不必先掌握全部包装语法。
它只在当前内核、当前函数调用中替换任务描述与 reset 检查，退出后恢复。
第一次 reset 后核对初始化索引、物理快照和语言；核对失败会在执行策略动作之前停止。
物理状态检查的绝对容差为 1e-8，仅用于数值一致性检查。

`eval_info.json` 是官方评测结果；`initial_audit.json` 和 `initial_state.pt` 是本项目增加的条件记录。
记录中的 task_id 对默认条件只是临时列表槽位，具体条件以 manifest 中的 task_name 与 condition 为准。
更新：已完成且 manifest 一致的回合直接读取结果，不重跑模型；配置不一致则要求新目录。
新增回合还会在 `select_action()` 前解码实际输入 tokens，保存 `policy_language_audit.json`。
**已完成的历史回合不会被补造 token 记录**；其语言证据仍是 `initial_audit.json` 与官方文本预处理调用链。
`public_loader()` 是一次调用内的接口适配：把旧 `torch_dtype` 关键字改成 `dtype`，值仍是 `bfloat16`；
不把整个策略强制转换精度。公开资源显式匿名读取，保留主动传入的 token。


### 11a．辅助采集定义（由助手维护，无需先读完）

只读采集前 120 个控制步（6 秒）的世界坐标、姿态、开合与物理子步接触；不改变模型输入、动作或环境步数。
左右指在一个控制区间内先后接触，与同一物理样本双侧接触分别记录；二者都不自动等于稳定抓取。
目标位置是目标 root body 原点；夹爪参考点来自 XML 的 grip_site，均不是保证的几何中心。
采集每 10 步落盘，函数结束时补存并归档。真实接口是否匹配，要先通过第 14 节的无策略检查。

接口依据：[Plus 目标 body 映射](https://github.com/sylvestf/LIBERO-plus/blob/4976dc30028e805ff8094b55501d532c48fec182/libero/libero/envs/bddl_base_domain.py#L377)、
[robosuite 1.4.1 左右指分组](https://github.com/ARISE-Initiative/robosuite/blob/v1.4.1/robosuite/models/grippers/panda_gripper.py#L26)、
[物理子步调用](https://github.com/ARISE-Initiative/robosuite/blob/v1.4.1/robosuite/environments/base.py#L357)。


In [ ]:
# 辅助采集：只读取仿真数据，不把目标真值加入策略输入，不修改动作。
import hashlib
import json
import shutil
from pathlib import Path
from unittest.mock import patch
import numpy as np
import torch


class GraspCapture:
    """记录前 120 个控制步；位置在控制边界采样，接触在 sim.step 子步中聚合。"""
    def __init__(self, env, output_dir, max_steps=120):
        self.env = env
        self.sim = env._env.sim
        self.core = env._env.env
        self.robot = self.core.robots[0]
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.max_steps = max_steps
        self.rows = []
        self.actions = []
        self.origin = float(self.data.time)
        model = self.sim.model
        target_name = "alphabet_soup_1"  # 来自当前基础 BDDL，不靠视频颜色猜测
        target = self.core.objects_dict[target_name]
        self.body_id = model.body_name2id(target.root_body)
        site_name = self.robot.gripper.important_sites["grip_site"]
        self.site_id = model.site_name2id(site_name)
        if self.site_id != self.robot.eef_site_id:
            raise RuntimeError("实际 eef_site_id 与 grip_site 不一致，停止采集")
        self.finger_indices = list(self.robot._ref_gripper_joint_pos_indexes)
        self.target_geoms = {model.geom_name2id(name) for name in target.contact_geoms}
        important = self.robot.gripper.important_geoms
        left = {model.geom_name2id(name) for name in important["left_finger"]}
        right = {model.geom_name2id(name) for name in important["right_finger"]}
        all_gripper = {model.geom_name2id(name) for name in self.robot.gripper.contact_geoms}
        if not self.target_geoms or not left or not right or left & right:
            raise RuntimeError("目标/左右指接触几何体为空或分组重叠，停止采集")
        self.groups = [left, right, all_gripper - left - right]
        self.metadata = {
            "target": target_name, "target_body": target.root_body,
            "gripper_site": site_name, "gripper_site_id": self.site_id,
            "gripper_class": type(self.robot.gripper).__name__,
            "finger_qpos_note": "actual joint coordinates, not a normalized action or commanded opening",
            "position_frame": "MuJoCo world", "position_unit": "m",
            "gripper_reference": "XML grip_site; not a measured finger midpoint",
            "object_reference": "object root body origin; not necessarily center of mass",
            "rotation": "world-from-site 3x3 matrix; object quaternion wxyz",
            "contact_columns": ["left_finger", "right_finger", "other_gripper"],
            "target_geom_names": [model.geom_id2name(i) for i in sorted(self.target_geoms)],
            "gripper_geom_groups": [[model.geom_id2name(i) for i in sorted(g)] for g in self.groups],
            "contact_rule": "target-vs-group contact with efc_address>=0 and dist<=0 at sampled physics states",
            "contact_scope": "initial boundary + after each sim.step; no contact force or grasp label inferred",
            "frame_alignment": "row 0 after reset/settling; row k after action k-1; action has one fewer row",
            "interval_alignment": "row k interval flags refer to action k-1; row 0 has no preceding interval",
            "control_frequency_hz": env.control_freq, "max_capture_steps": max_steps,
        }
        self.append(np.zeros(3, dtype=bool), False, 0)
        self.flush("initialized")

    @property
    def data(self):
        return getattr(self.sim.data, "_data", self.sim.data)

    def contacts(self):
        flags = np.zeros(3, dtype=bool)
        for i in range(int(self.data.ncon)):
            contact = self.data.contact[i]
            if contact.efc_address < 0 or contact.dist > 0:
                continue
            a, b = int(contact.geom1), int(contact.geom2)
            other = b if a in self.target_geoms else a if b in self.target_geoms else None
            if other is not None:
                for j, group in enumerate(self.groups):
                    flags[j] |= other in group
        return flags

    def append(self, interval_flags, both_same_sample, physics_samples):
        data = self.data
        row = {
            "frame_index": np.int64(len(self.rows)),
            "elapsed_s": np.float64(data.time - self.origin),
            "simulation_time_s": np.float64(data.time),
            "gripper_xyz": np.array(data.site_xpos[self.site_id], copy=True),
            "gripper_rotation_world": np.array(data.site_xmat[self.site_id], copy=True).reshape(3, 3),
            "object_xyz": np.array(data.xpos[self.body_id], copy=True),
            "object_quat_wxyz": np.array(data.xquat[self.body_id], copy=True),
            "finger_qpos": np.array(data.qpos[self.finger_indices], copy=True),
            "contact_now": self.contacts(),
            "contact_during_step": interval_flags.copy(),
            "both_fingers_same_physics_sample": np.bool_(both_same_sample),
            "physics_samples": np.int64(physics_samples),
        }
        if not all(np.isfinite(value).all() for value in row.values()):
            raise RuntimeError("采集发现非有限仿真状态，请检查环境")
        self.rows.append(row)

    def step(self, original_step, env, action):
        if env is not self.env:
            raise RuntimeError("当前采集仅支持一个环境实例")
        if len(self.actions) >= self.max_steps:
            return original_step(env, action)
        executed = np.array(action, copy=True)  # 环境实际收到的后处理动作
        interval = self.contacts()
        both = bool(interval[0] and interval[1])
        samples = 0
        original_sim_step = self.sim.step

        def sim_step_with_contacts(*args, **kwargs):
            nonlocal both, samples
            result = original_sim_step(*args, **kwargs)
            flags = self.contacts()
            interval[:] |= flags
            both |= bool(flags[0] and flags[1])
            samples += 1
            return result

        # 只在当前控制步中包装一次；异常也会恢复 sim.step。
        with patch.object(self.sim, "step", sim_step_with_contacts):
            result = original_step(env, action)
        if samples == 0:
            raise RuntimeError("本控制步没有经过 sim.step，不能声称已采集物理子步接触")
        self.actions.append(executed)
        self.append(interval, both, samples)
        if len(self.actions) % 10 == 0:
            self.flush("collecting")
        return result

    def flush(self, status):
        tensors = {
            key: torch.from_numpy(np.stack([row[key] for row in self.rows]))
            for key in self.rows[0]
        }
        tensors["executed_action"] = torch.from_numpy(
            np.stack(self.actions) if self.actions else np.empty((0, 7), dtype=np.float64)
        )
        # 原子替换，减少中断时出现半个 .pt 文件的风险。
        temporary = self.output_dir / "grasp_trace.tmp.pt"
        torch.save(tensors, temporary)
        temporary.replace(self.output_dir / "grasp_trace.pt")
        meta = {**self.metadata, "status": status, "recorded_control_steps": len(self.actions)}
        (self.output_dir / "grasp_schema.json").write_text(
            json.dumps(meta, ensure_ascii=False, indent=2) + "\n"
        )


def archive_camera_run(source, destination):
    """可续传同内容文件；若云盘已有不同内容则停止，保留双方数据。"""
    source, destination = Path(source), Path(destination)
    files = [path for path in source.rglob("*") if path.is_file()]
    for path in files:
        target = destination / path.relative_to(source)
        if target.exists() and hashlib.sha256(path.read_bytes()).digest() != hashlib.sha256(target.read_bytes()).digest():
            raise FileExistsError(f"归档存在不同内容，不覆盖：{target}；请选择新实验目录")
    for path in files:
        target = destination / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            temporary = target.with_name(target.name + ".copying")
            shutil.copy2(path, temporary)
            temporary.replace(target)
    print("本回合文件已核对并归档：", destination)


In [ ]:
import gc
import time
from contextlib import ExitStack
from functools import wraps
from importlib.metadata import version
from unittest.mock import patch
from transformers import AutoModelForImageTextToText, AutoProcessor, AutoTokenizer
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, VLAFlowMatching
from lerobot.scripts import lerobot_eval


def public_loader(loader, *, normalize_dtype=False):
    """仅在本次公开模型加载中适配旧关键字；不改变实际计算精度。"""
    @wraps(loader)
    def load(*args, **kwargs):
        if normalize_dtype and "torch_dtype" in kwargs:
            old_dtype = kwargs.pop("torch_dtype")
            if "dtype" in kwargs and kwargs["dtype"] != old_dtype:
                raise ValueError("torch_dtype 与 dtype 设置不一致")
            kwargs["dtype"] = old_dtype
        kwargs.setdefault("token", False)  # 避免 VS Code 会话访问 Colab UI 密钥接口
        return loader(*args, **kwargs)
    return load


def controlled_camera_task(condition):
    condition_ids = {"default_camera": 653, "view_1": 653, "view_5": 656, "view_10": 661, "view_15": 666}
    task_id = condition_ids[condition]
    slot_name = benchmark.libero_task_map["libero_object"][task_id]
    original_task = benchmark.task_maps["libero_object"][slot_name]
    assert original_task.name.split("_view_", 1)[0] == pilot_base_task
    if condition != "default_camera":
        expected_name = f"{pilot_base_task}_{condition}_15_100_0_0_initstate_0"
        if original_task.name != expected_name:
            raise RuntimeError(f"任务 ID 与完整名称不符：{task_id}: {original_task.name}")

    if condition == "default_camera":
        selected_task = original_task._replace(
            name=pilot_base_task, language=pilot_instruction,
            bddl_file=pilot_base_task + ".bddl", init_states_file=pilot_base_task + ".pruned_init",
        )
    else:
        selected_task = original_task._replace(language=pilot_instruction)

    return task_id, slot_name, original_task, selected_task


def run_controlled_camera_pilot(condition, *, seed=1000, capture_grasp=False,
                                run_root=None, archive_root=None, n_action_steps=None):
    if not isinstance(seed, int) or seed < 0:
        raise ValueError("seed 必须是非负整数")
    if n_action_steps is not None:
        if type(n_action_steps) is not int or not 1 <= n_action_steps <= 50:
            raise ValueError("本对照的 n_action_steps 必须为 1–50 的整数")
        if not capture_grasp:
            raise ValueError("执行窗口对照需 capture_grasp=True，以便记录控制步与模型调用")
    task_id, slot_name, original_task, selected_task = controlled_camera_task(condition)
    root = Path(run_root) if run_root is not None else PILOT_ROOT
    output_dir = root / condition
    if n_action_steps is not None:
        output_dir = root / f"n_action_steps_{n_action_steps}" / condition
    if capture_grasp:
        output_dir = output_dir / f"seed_{seed}"
    manifest = {
        "condition": condition, "task_id_slot": task_id, "task_name": selected_task.name,
        "original_slot_task_name": original_task.name, "effective_language": pilot_instruction,
        "checkpoint": CHECKPOINT, "checkpoint_revision": checkpoint_revision,
        "libero_plus_revision": PLUS_SHA, "asset_revision": ASSET_REVISION,
        "seed": seed, "episodes": 1, "reference_audit": str(AUDIT_DIR / "audit.json"),
        "protocol": "controlled_base_bddl_language_grasp_v1" if capture_grasp else "controlled_base_bddl_language",
    }
    if capture_grasp:
        manifest["capture_steps"] = 120
    if n_action_steps is not None:
        manifest.update({
            "n_action_steps": n_action_steps, "chunk_size": 50, "num_steps": 10,
            "timing_protocol": "get_action_chunk_synchronized_wall_v1",
        })
    manifest_path = output_dir / "manifest.json"
    if manifest_path.exists():
        saved_manifest = json.loads(manifest_path.read_text())
        changed = [key for key, value in manifest.items() if saved_manifest.get(key) != value]
        if changed:
            raise RuntimeError(f"已有记录的配置不一致：{changed}；请使用新的 PILOT_ROOT")
    if (output_dir / "eval_info.json").exists():
        if not manifest_path.exists():
            raise RuntimeError("已有结果但缺少 manifest，不能自动复用；请先检查目录")
        audit = json.loads((output_dir / "initial_audit.json").read_text())
        if not (audit["language_matches"] and all(audit["state_matches_reference"].values())):
            raise RuntimeError("已有结果的初始条件检查未通过，不能自动复用")
        metrics = json.loads((output_dir / "eval_info.json").read_text())
        if capture_grasp:
            for name in ["grasp_trace.pt", "grasp_schema.json", "first_noise.pt", "noise_metadata.json", "policy_language_audit.json"]:
                if not (output_dir / name).is_file():
                    raise RuntimeError(f"已有回合缺少新增采集文件 {name}；请使用新的实验目录")
            if json.loads((output_dir / "grasp_schema.json").read_text())["status"] != "evaluation_returned":
                raise RuntimeError("已有采集未正常结束，不能作为完整回合复用")
        if n_action_steps is not None:
            config_file, timing_file = output_dir / "actual_policy_config.json", output_dir / "timing.json"
            if not config_file.is_file() or not timing_file.is_file():
                raise RuntimeError("已有结果缺少执行配置或计时，不能自动作为本对照复用")
            actual = json.loads(config_file.read_text())
            if any(actual.get(k) != v for k, v in {
                "n_action_steps": n_action_steps, "chunk_size": 50, "num_steps": 10,
            }.items()):
                raise RuntimeError("已有实际策略配置与本次对照不符")
        if archive_root is not None:
            archive_camera_run(output_dir, Path(archive_root) / output_dir.relative_to(root))
        print("复用已完成结果，不重新执行模型：", condition, metrics["overall"])
        if not (output_dir / "policy_language_audit.json").exists():
            print("历史回合已有环境语言检查，但没有新增的策略 token 解码记录。")
        return metrics

    output_dir.mkdir(parents=True, exist_ok=True)
    manifest["runtime_versions"] = {
        name: version(name) for name in ["torch", "transformers", "tokenizers", "IPython", "jupyter-client"]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))

    original_reset = LiberoEnv.reset
    reset_checked = False
    policy_language_checked = False
    grasp_capture = None
    noise_recorded = False
    env_steps = 0
    chunk_timings = []
    original_get_action_chunk = SmolVLAPolicy._get_action_chunk
    original_make_policy = lerobot_eval.make_policy
    original_step = LiberoEnv.step
    original_sample_noise = VLAFlowMatching.sample_noise

    def step_with_grasp_capture(env, action):
        nonlocal env_steps
        if grasp_capture is None:
            raise RuntimeError("抓取采集器尚未初始化")
        result = grasp_capture.step(original_step, env, action)
        env_steps += 1
        return result

    def make_policy_with_config_audit(*args, **kwargs):
        policy = original_make_policy(*args, **kwargs)
        actual = {name: getattr(policy.config, name) for name in ["n_action_steps", "chunk_size", "num_steps"]}
        expected = {"n_action_steps": n_action_steps, "chunk_size": 50, "num_steps": 10}
        if actual != expected:
            raise RuntimeError(f"实际策略配置不符：{actual}，预期 {expected}")
        if policy.config.rtc_config is not None:
            raise RuntimeError("此执行窗口对照仅支持当前普通动作队列路径")
        if policy._queues["action"].maxlen != n_action_steps:
            raise RuntimeError("动作队列容量未随配置更新，停止执行")
        (output_dir / "actual_policy_config.json").write_text(json.dumps(actual, indent=2))
        return policy

    def get_action_chunk_with_timing(policy, *args, **kwargs):
        parameter = next(policy.parameters(), None)
        device = parameter.device if parameter is not None else torch.device("cpu")
        def synchronize():
            if device.type == "cuda":
                torch.cuda.synchronize(device)
        synchronize()
        started = time.perf_counter()
        completed = False
        try:
            result = original_get_action_chunk(policy, *args, **kwargs)
            synchronize()
            completed = True
            return result
        finally:
            chunk_timings.append({
                "env_steps_before_chunk": env_steps,
                "wall_s": time.perf_counter() - started,
                "completed": completed,
            })

    def sample_noise_with_record(model, *args, **kwargs):
        nonlocal noise_recorded
        noise = original_sample_noise(model, *args, **kwargs)
        if not noise_recorded:
            saved = noise.detach().cpu().contiguous().clone()
            torch.save(saved, output_dir / "first_noise.pt")
            (output_dir / "noise_metadata.json").write_text(json.dumps({
                "shape": list(saved.shape), "dtype": str(saved.dtype),
                "sha256": hashlib.sha256(saved.view(torch.uint8).numpy().tobytes()).hexdigest(),
                "scope": "actual first sampled noise; recorded without replacing it",
            }, indent=2))
            noise_recorded = True
        return noise

    original_select_action = SmolVLAPolicy.select_action

    def select_action_with_language_audit(policy, batch, *args, **kwargs):
        nonlocal policy_language_checked
        if not policy_language_checked:
            token_ids = batch["observation.language.tokens"].detach().cpu()
            token_mask = batch["observation.language.attention_mask"].detach().cpu().bool()
            tokenizer = policy.model.vlm_with_expert.processor.tokenizer
            decoded = [
                tokenizer.decode(ids[mask].tolist(), skip_special_tokens=True).strip()
                for ids, mask in zip(token_ids, token_mask, strict=True)
            ]
            matches = all(text == pilot_instruction.strip() for text in decoded)
            (output_dir / "policy_language_audit.json").write_text(json.dumps({
                "expected_instruction": pilot_instruction, "decoded_instruction": decoded,
                "token_ids": token_ids.tolist(), "attention_mask": token_mask.tolist(),
                "matches": matches,
            }, ensure_ascii=False, indent=2))
            if not matches:
                raise RuntimeError("策略实际收到的语言与共同指令不一致，已在动作生成前停止")
            policy_language_checked = True
            print("策略实际输入的指令：", decoded, flush=True)
        return original_select_action(policy, batch, *args, **kwargs)


    def reset_with_audit(self, seed=None, **kwargs):
        nonlocal reset_checked, grasp_capture
        init_index = self.init_state_id % len(self._init_states)
        selected_init = torch.from_numpy(np.asarray(self._init_states[init_index]).copy())
        observation, info = original_reset(self, seed=seed, **kwargs)
        if reset_checked:
            return observation, info
        raw_data = getattr(self._env.sim.data, "_data", self._env.sim.data)
        state = {
            key: torch.from_numpy(np.array(getattr(raw_data, key), copy=True))
            for key in ["qpos", "qvel", "act", "mocap_pos", "mocap_quat"]
        }
        state["selected_init"] = selected_init
        matches = {}
        max_differences = {}
        for key, reference in pilot_reference.items():
            same_shape = state[key].shape == reference.shape
            matches[key] = same_shape and torch.allclose(state[key], reference, rtol=0, atol=1e-8)
            max_differences[key] = (
                (state[key] - reference).abs().max().item()
                if same_shape and state[key].numel() else None
            )
        camera_id = self._env.sim.model.camera_name2id("agentview")
        language_ok = self.task_description == self._env.language_instruction == pilot_instruction
        initial_audit = {
            "condition": condition, "task_name": self.task, "effective_language": self.task_description,
            "base_bddl_language": self._env.language_instruction, "language_matches": language_ok,
            "init_state_index": int(init_index), "state_matches_reference": matches,
            "state_max_abs_differences": max_differences,
            "agentview_world_position": np.asarray(raw_data.cam_xpos[camera_id]).tolist(),
            "agentview_world_rotation": np.asarray(raw_data.cam_xmat[camera_id]).reshape(3, 3).tolist(),
            "agentview_fovy": float(self._env.sim.model.cam_fovy[camera_id]),
            "robot": type(self._env.robots[0].robot_model).__name__,
            "control_mode": self.control_mode, "control_frequency": self.control_freq,
            "settling_steps": self.num_steps_wait, "initial_success": bool(self._env.check_success()),
        }
        torch.save(state, output_dir / "initial_state.pt")
        (output_dir / "initial_audit.json").write_text(json.dumps(initial_audit, ensure_ascii=False, indent=2))
        settings_ok = (
            init_index == 0 and self.is_libero_plus and self.control_mode == "relative"
            and self.control_freq == 20 and self.num_steps_wait == 10
            and initial_audit["robot"] == pilot_audit["tasks"][0]["robot"]
            and not initial_audit["initial_success"]
        )
        if not (language_ok and settings_ok and all(matches.values())):
            raise RuntimeError(f"初始条件未通过核对，先检查 {output_dir / 'initial_audit.json'}")
        if capture_grasp:
            grasp_capture = GraspCapture(self, output_dir, max_steps=120)
        reset_checked = True
        print("已通过动作执行前的初始条件核对：", condition, flush=True)
        return observation, info

    arguments = [
        "lerobot-eval", f"--policy.path={checkpoint_snapshot}", "--policy.device=cuda",
        "--env.type=libero_plus", "--env.task=libero_object", f"--env.task_ids=[{task_id}]",
        "--env.observation_height=256", "--env.observation_width=256", "--env.fps=20",
        "--env.control_mode=relative", "--env.init_states=true", "--env.hard_reset=true",
        '--env.camera_name_mapping={"agentview_image":"camera1","robot0_eye_in_hand_image":"camera2"}',
        "--env.max_parallel_tasks=1", "--eval.batch_size=1", "--eval.n_episodes=1",
        "--eval.use_async_envs=false", f"--seed={seed}", f"--job_name=smolvla_camera_{condition}",
        f"--output_dir={output_dir}",
    ]
    if n_action_steps is not None:
        arguments += [f"--policy.n_action_steps={n_action_steps}", "--policy.chunk_size=50", "--policy.num_steps=10"]
    benchmark.task_maps["libero_object"][slot_name] = selected_task
    run_started = time.perf_counter()
    evaluation_returned = False
    archive_problem = None
    try:
        # 所有适配仅在本次调用内生效，正常结束或异常都会恢复原方法。
        with ExitStack() as stack:
            stack.enter_context(patch.object(LiberoEnv, "reset", reset_with_audit))
            stack.enter_context(patch.object(SmolVLAPolicy, "select_action", select_action_with_language_audit))
            stack.enter_context(patch.object(sys, "argv", arguments))
            if n_action_steps is not None:
                stack.enter_context(patch.object(lerobot_eval, "make_policy", make_policy_with_config_audit))
                stack.enter_context(patch.object(SmolVLAPolicy, "_get_action_chunk", get_action_chunk_with_timing))
            if capture_grasp:
                stack.enter_context(patch.object(LiberoEnv, "step", step_with_grasp_capture))
                stack.enter_context(patch.object(VLAFlowMatching, "sample_noise", sample_noise_with_record))
            for loader_class in [AutoModelForImageTextToText, AutoProcessor, AutoTokenizer]:
                stack.enter_context(patch.object(
                    loader_class, "from_pretrained",
                    public_loader(
                        loader_class.from_pretrained,
                        normalize_dtype=loader_class is AutoModelForImageTextToText,
                    ),
                ))
            lerobot_eval.main()
            evaluation_returned = True
    except Exception as error:
        (output_dir / "run_error.json").write_text(json.dumps({
            "error": f"{type(error).__name__}: {error}",
        }, ensure_ascii=False, indent=2))
        raise
    finally:
        benchmark.task_maps["libero_object"][slot_name] = original_task
        if grasp_capture is not None:
            grasp_capture.flush("evaluation_returned" if evaluation_returned else "interrupted")
        if n_action_steps is not None:
            completed_chunks = [item["wall_s"] for item in chunk_timings if item["completed"]]
            timing = {
                "protocol": "get_action_chunk_synchronized_wall_v1",
                "evaluation_returned": evaluation_returned,
                "env_steps": env_steps, "simulated_control_s": env_steps / 20,
                "n_action_steps": n_action_steps, "execution_window_s": n_action_steps / 20,
                "chunk_calls": len(chunk_timings),
                "chunk_wall_s_total": sum(completed_chunks),
                "chunk_wall_s_mean": sum(completed_chunks) / len(completed_chunks) if completed_chunks else None,
                "run_wall_s_including_loading_and_setup": time.perf_counter() - run_started,
                "chunks": chunk_timings,
                "note": "Chunk wall time includes its preparation and synchronized GPU work; excludes policy loading. Not a real-time deployment claim.",
            }
            (output_dir / "timing.json").write_text(json.dumps(timing, indent=2))
        if archive_root is not None:
            try:
                archive_camera_run(output_dir, Path(archive_root) / output_dir.relative_to(root))
            except Exception as error:
                archive_problem = error
                print("归档未完成；本地结果保留：", output_dir, str(error), flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if archive_problem is not None:
        raise RuntimeError("评测结果已保存，但归档失败；先处理归档再继续下一回合") from archive_problem
    if capture_grasp and (grasp_capture is None or not noise_recorded):
        raise RuntimeError("评测返回但缺少实际状态或噪声采集，请检查执行入口")
    metrics = json.loads((output_dir / "eval_info.json").read_text())
    print(condition, metrics["overall"])
    return metrics

## 12．先执行默认相机基线，看一次真实任务视频

本节会加载完整 SmolVLA 并执行一回合任务。
先判断模型在默认条件下是否能完成该操作，再解释其他视角的表现。
若失败，请先给出你对失败阶段的判断：接近目标、抓取、搬运、放置，或者环境/输入错误。
不要只根据 `pc_success` 一个数决定修改算法。
已保存的历史输出：本回合成功，之后才在 IPython 的 `Video(filename=...)` 处报错。
这里保留原输出作为排错证据，不代表修正后的单元已经执行。
重放视频直接执行下方 **12a**，不需要重新加载模型。


In [ ]:
baseline_metrics = run_controlled_camera_pilot("default_camera")
# 播放视频请运行下面独立的“12a”单元；播放器异常无需重新评测。


### 12a．只读取已有结果和播放视频（不运行模型）

这个单元独立于第 11 节的模型加载，不修改结果文件。会话仍在时直接运行；新会话则先从 Drive 恢复实验目录并调整路径。
当前已保存输出：默认条件 1/1 成功，view_1 与 view_15 各 0/1；三组都是单回合试运行。
`Video` 使用位置参数传路径，避开当前 IPython 对 `filename=` 的空 `data` 处理错误。
先比较默认条件与 view_1：各自在哪一帧接近、闭合夹爪、抬起、放置？失败从哪个阶段开始？


In [ ]:
import json
from pathlib import Path
from IPython.display import Video, display

# 若从 Drive 重放，改成 /content/drive/MyDrive/robot-lab/experiments/相应实验目录。
REPLAY_ROOT = Path("/content/outputs/smolvla_camera_pilot_001")

for condition in ["default_camera", "view_1", "view_15"]:
    condition_dir = REPLAY_ROOT / condition
    result_file = condition_dir / "eval_info.json"
    if not result_file.is_file():
        print(condition, "尚无完整结果：", result_file)
        continue
    result = json.loads(result_file.read_text())
    print(condition, "回合数：", result["overall"]["n_episodes"],
          "成功百分比：", result["overall"]["pc_success"])
    video_paths = result["overall"].get("video_paths", [])
    if not video_paths:
        print("结果未记录视频路径，请检查：", condition_dir / "videos")
    for video_path in video_paths:
        if not video_path:
            print("结果中的视频路径为空，请检查：", condition_dir / "videos")
            continue
        saved_path = Path(video_path)
        # 官方保存绝对路径；迁移到 Drive 后改为当前条件目录下的同一个 videos 相对路径。
        if "videos" not in saved_path.parts:
            raise ValueError(f"未识别的视频目录结构：{video_path}")
        video = condition_dir.joinpath(*saved_path.parts[saved_path.parts.index("videos"):])
        if not video.is_file():
            print("未找到视频（评测结果保留）：", video)
            continue
        display(Video(str(video), embed=True))


### 默认条件核对完成后，再执行两个视角条件

保留用户已启用的两次调用。修正后的入口会核对 manifest，并复用已完成结果，避免重复计算。
若要真正重复评测，请在第 11 节选择新的 `PILOT_ROOT`，保留旧实验。
本次已保存输出：view_1、view_15 各一回合未成功；先在 12a 查看视频，区分接近、抓取、搬运和放置阶段。
后续正式结论需要更多配对重复和未参与选择的视角，不能把单回合差异称为稳定成功率下降。


In [ ]:
view_1_metrics = run_controlled_camera_pilot("view_1")
view_15_metrics = run_controlled_camera_pilot("view_15")

### 当前警告如何处理

| 来源 | 原因与处理 |
| --- | --- |
| `torch_dtype` 弃用 | LeRobot 旧调用关键字与 Transformers 5.x 的接口差异；本次调用中转为 `dtype`，精度值不变。 |
| Colab `HF_TOKEN` secret 超时 | VS Code 不具备 Colab 网页密钥交互；公开下载和模型加载显式 `token=False`。匿名限速提示仍可能出现，不是推理异常。 |
| robosuite private macros | 第 4 节按[官方 setup_macros](https://github.com/ARISE-Initiative/robosuite/blob/v1.4.1/robosuite/scripts/setup_macros.py)建立默认私有配置，保留已有配置；已导入的当前会话可能需下次启动才生效。 |
| `datetime.utcnow()` | 旧 jupyter-client 的通信时间戳调用；下次安装使用 8.10.0，安装后需重启内核。与机器人动作时间无关。 |
| Dopamine / Gym 依赖冲突 | 专用会话移除不用的 `dopamine-rl`，保留 Plus 所需 `gym==0.26.2`。 |
| Gym 停止维护 / NumPy 2 提示 | Plus 仍依赖旧 Gym API。本次真实回合已跑通，但不代表所有 Gym 环境都兼容 NumPy 2；不盲换 Gymnasium 或降级 LeRobot 需要的 NumPy。 |
| job name 缺省 | 启动参数显式提供条件对应的 `job_name`。 |

没有使用全局忽略警告。这些修改已做本地接口检查，**更新后的整套 Colab 安装和 GPU 评测仍需实际验证**。
当前先运行 12a 重放已有视频，不为清理历史警告重跑模型。Notebook 的旧输出保留，不会随源代码修改自动消失。


### 保存闭环试运行产物

任务执行后，保存官方结果、视频和条件审计文件。只保存实验产物，不复制模型权重或扩展资源。

In [ ]:
assert (PILOT_ROOT / "default_camera" / "eval_info.json").is_file()
pilot_destination = Path("/content/drive/MyDrive/robot-lab/experiments") / PILOT_ROOT.name
assert Path("/content/drive/MyDrive").is_dir(), "先使用第 9 节挂载云盘"
assert not pilot_destination.exists(), "已有同名归档，请换新名称或明确需要合并的内容"
shutil.copytree(PILOT_ROOT, pilot_destination)
print("闭环试运行产物已保存：", pilot_destination)

## 14．首次抓取配对采集：先查接口，再跑一对

用户的视频标注已完成。默认条件也有接近接触，因此不把“任意接触”直接判为失败。
先用同一初始物理状态比较默认条件与 view_15；这轮保持 checkpoint、指令、控制和队列设置。
第 14a 单元只创建并 reset 环境，不加载模型，提前验证目标和夹爪接口。

已有会话下不重装；新会话按顶部路线恢复。默认第 14b 只跑 seed=1000 的一对。
检查文件保存后，再执行第 14c 的另外两对；每个 seed 都核对同一物理初始状态。
相同 seed 不自动作为“噪声相同”的证据：代码记录实际 first_noise，配对后比较哈希。
本轮只检查同一初始化下的随机重复，不代表跨初始化或跨任务泛化统计。


In [ ]:
# 14a：不加载策略，先验证真实仿真接口；产物保存在独立检查目录。
import io
from contextlib import redirect_stdout

GRASP_ROOT = Path("/content/outputs/smolvla_grasp_20261031_001")
GRASP_ARCHIVE = Path("/content/drive/MyDrive/robot-lab/experiments") / GRASP_ROOT.name
assert Path("/content/drive/MyDrive").is_dir(), "先完成 10a 云盘挂载"

GRASP_INTERFACES_VERIFIED = False

def check_grasp_interfaces():
    task_id, slot, original, selected = controlled_camera_task("view_15")
    env = None
    check_dir = GRASP_ROOT / "interface_check"
    check_dir.mkdir(parents=True, exist_ok=True)
    benchmark.task_maps["libero_object"][slot] = selected
    try:
        log = io.StringIO()
        with redirect_stdout(log):
            suite = benchmark.get_benchmark_dict()["libero_object"]()
        (check_dir / "suite_creation.log").write_text(log.getvalue())
        env = LiberoEnv(
            task_suite=suite, task_id=task_id, task_suite_name="libero_object",
            observation_height=256, observation_width=256, obs_type="pixels_agent_pos",
            camera_name_mapping={"agentview_image": "camera1", "robot0_eye_in_hand_image": "camera2"},
            init_states=True, episode_index=0, n_envs=1, num_steps_wait=10,
            control_freq=20, control_mode="relative", is_libero_plus=True, hard_reset=True,
        )
        env.reset(seed=1000)
        collector = GraspCapture(env, check_dir)
        # 只在这个独立检查环境中执行一个官方无操作动作，验证物理子步包装确实被调用。
        from lerobot.envs.libero import get_libero_dummy_action
        collector.step(LiberoEnv.step, env, np.asarray(get_libero_dummy_action(), dtype=np.float64))
        collector.flush("interface_check_only")
        print("接口检查通过；目标：", collector.metadata["target_body"])
        print("夹爪参考点：", collector.metadata["gripper_site"])
        print("左右指/其他夹爪几何体：", collector.metadata["gripper_geom_groups"])
        print("检查环境已额外 step，随后关闭；正式评测将重新创建并核对初始状态。")
    finally:
        benchmark.task_maps["libero_object"][slot] = original
        if env is not None:
            env.close()

check_grasp_interfaces()
GRASP_INTERFACES_VERIFIED = True


In [ ]:
# 14b：先执行一对，记录数据并逐回合保存到 Drive。
def run_grasp_pair(seed):
    assert GRASP_INTERFACES_VERIFIED, "先执行并通过 14a 接口检查"
    results = {}
    for condition in ["default_camera", "view_15"]:
        results[condition] = run_controlled_camera_pilot(
            condition, seed=seed, capture_grasp=True,
            run_root=GRASP_ROOT, archive_root=GRASP_ARCHIVE,
        )
    hashes = [json.loads((GRASP_ROOT / condition / f"seed_{seed}" / "noise_metadata.json").read_text())["sha256"]
              for condition in ["default_camera", "view_15"]]
    if hashes[0] != hashes[1]:
        raise RuntimeError("配对第一份实际噪声不同；保留结果，先排查随机数来源再继续")
    print("第一份实际噪声一致；此后闭环状态可以分叉，不能声称所有后续噪声/轨迹都相同。")
    for condition, result in results.items():
        print(seed, condition, result["overall"]["pc_success"])
    return results

first_pair = run_grasp_pair(1000)


In [ ]:
# 14c：确认第一对文件和云盘归档正常后再执行。默认不自动扩大实验。
for seed in [1001, 1002]:
    run_grasp_pair(seed)


## 15．你的核心实践：从真实位置计算相对位置

采集成功后，把 GRASP_ARCHIVE 指向的实验文件夹下载到本地 outputs，或在当前 Colab 直接读取。
只需补下面函数的 TODO；不需要默写采集框架或重新理解整个 Transformer。

- 输入：两个相同 shape 的位置 Tensor，例如 [121,3]，依次是每个采样时刻的世界坐标（米）。
- 输出：相同 shape 的相对位置，约定为“夹爪参考点减目标参考点”。
- 请先自己写相减和 shape 检查，再让助手审查。
- 比较首次闭合附近的三轴偏差，并结合 contact_during_step、finger_qpos 和物体抬升观察。
- 不以距离为零或任意接触作为成功判据；目标 root 原点、grip_site 都有定义上的偏置。

数组约定：frame_index=0 是稳定等待后的初始帧；第 k 行在第 k 个动作完成后。
executed_action 只有 T-1 行，第 k-1 个动作对应第 k 行状态。
contact_during_step[k] 是该动作区间各物理样本接触的并集；
both_fingers_same_physics_sample[k] 才表示采样到过同一物理样本中的双侧接触，但仍不等于稳定抓取。


In [ ]:
def relative_position(gripper_xyz, object_xyz):
    """返回同一世界坐标系中的夹爪参考点减目标参考点，单位米。"""
    # TODO（你来写）：检查两个 Tensor 的 shape 相同，末维为 3，然后返回相对位置。
    if gripper_xyz.shape != object_xyz.shape:
        raise ValueError("gripper_xyz and object_xyz must have the same shape")
    if gripper_xyz.shape[-1] != 3:
        raise ValueError("gripper_xyz and object_xyz must have 3 dimensions in the last axis")

    return gripper_xyz - object_xyz


In [ ]:
# 采集完成后运行；本地分析时把 ANALYSIS_ROOT 换成下载后的实际目录。
from pathlib import Path
import torch
ANALYSIS_ROOT = Path("/content/outputs/smolvla_grasp_20261031_001")
trace_path = ANALYSIS_ROOT / "view_15" / "seed_1000" / "grasp_trace.pt"
assert trace_path.is_file(), "真实位置数据尚未采集或路径未设置，不要用旧视频代替位置日志"
trace = torch.load(trace_path, map_location="cpu", weights_only=True)
print({name: tuple(value.shape) for name, value in trace.items()})
relative_xyz = relative_position(trace["gripper_xyz"], trace["object_xyz"])
print(relative_xyz.shape)
print(relative_xyz[40:61])  # 20 Hz 下约 2–3 秒，不要求提前知道具体数值


## 16．把首次双侧接触整理成真实结果表（CPU 可运行）

你只实现 first_true_time() 的核心几行。输入 time、mask 均为一维且等长；mask 是 bool。
返回第一个 True 对应的 Python float；没有 True 时返回 None。输入检查由助手准备。

提示顺序：用 mask.nonzero() 找位置 → 判断是否为空 → 取首个位置 → 查询 time 并转成 Python 数值。
不需要 for 循环逐个比较时间。这里“首次时间”是检测到接触的控制区间末时刻，精度约 0.05 秒。
下面的读取与 CSV 保存由助手准备；函数未完成时会明确报 NotImplementedError，不会填造结果。


In [ ]:
def first_true_time(time, mask):
    """返回首个 True 对应的时间（Python float）；全 False 或空输入返回 None。"""
    if not isinstance(time, torch.Tensor) or not isinstance(mask, torch.Tensor):
        raise TypeError("time 和 mask 都需要是 Tensor")
    if time.ndim != 1 or mask.ndim != 1 or time.shape != mask.shape:
        raise ValueError("time 和 mask 必须是一维且等长")
    if mask.dtype != torch.bool:
        raise TypeError("mask 需要是 torch.bool")
    if time.device != mask.device:
        raise ValueError("time 和 mask 需要在同一设备")
    # TODO 1：找出 mask 中 True 的位置。
    true_indices = mask.nonzero(as_tuple=False).squeeze(-1)
    if true_indices.numel() == 0:
        return None

    # TODO 2：没有 True 时返回 None。
    # TODO 3：取第一个 True 对应的时间，并转成 Python float。

    return time[true_indices[0]].item()


In [ ]:
from pathlib import Path
import csv
import json
import torch

# 本地已同步这轮数据；Colab 则从 /content/outputs 或 Drive 自动定位。
candidates = [
    Path("/content/outputs/smolvla_grasp_20261031_001"),
    Path("/content/drive/MyDrive/robot-lab/experiments/smolvla_grasp_20261031_001"),
    Path.cwd() / "outputs/smolvla_grasp_20261031_001",
    Path.cwd().parent / "outputs/smolvla_grasp_20261031_001",
]
SUMMARY_ROOT = next((root for root in candidates if (root / "view_15/seed_1000/grasp_trace.pt").is_file()), None)
assert SUMMARY_ROOT is not None, "未找到这轮真实日志，请核对数据目录"


def summarize_grasp_run(run_dir):
    run_dir = Path(run_dir)
    trace = torch.load(run_dir / "grasp_trace.pt", map_location="cpu", weights_only=True)
    metrics = json.loads((run_dir / "eval_info.json").read_text())
    manifest = json.loads((run_dir / "manifest.json").read_text())
    return {
        "condition": manifest["condition"], "seed": manifest["seed"],
        "n_action_steps": manifest.get("n_action_steps", 50),
        "success": bool(metrics["overall"]["pc_success"] == 100),
        "first_bilateral_interval_end_s": first_true_time(
            trace["elapsed_s"], trace["both_fingers_same_physics_sample"]),
        "max_object_rise_first_6s_cm": float((trace["object_xyz"][:, 2] - trace["object_xyz"][0, 2]).max()) * 100,
        "captured_control_steps": len(trace["executed_action"]),
    }


def save_grasp_summary(rows, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)
    print("汇总表已保存：", csv_path)


rows = [summarize_grasp_run(SUMMARY_ROOT / condition / f"seed_{seed}")
        for seed in [1000, 1001, 1002] for condition in ["default_camera", "view_15"]]
# 这是上轮真实日志的核对参考，不用于给函数填造输出。
expected = [2.80, None, 3.00, 3.30, 2.80, None]
for row, reference in zip(rows, expected, strict=True):
    observed = row["first_bilateral_interval_end_s"]
    assert observed is None or isinstance(observed, float), "请返回 Python float（可使用 .item()），不是 Tensor"
    assert (observed is None) if reference is None else (observed is not None and abs(observed - reference) < 1e-6)
save_grasp_summary(rows, SUMMARY_ROOT / "analysis/first_grasp_summary.csv")


## 17．执行窗口 50/10 的最小对照框架（手动启动 GPU）

研究假设：更频繁重新观察和预测，能否在离开目标前修正不对称接触？这是待验证假设。
50 步 / 20 Hz = 2.5 秒仿真执行窗口；10 步 / 20 Hz = 0.5 秒。动作块仍预测 50 步，积分仍为 10 步。
比较成功、首次双侧接触、抬升与计算代价，不把改善直接叫作“视觉编码器被修复”。

**当前会话启动：**重新执行第 11 节的评测函数定义（已增加参数、真实配置检查与计时），
运行下面 17a 的定义单元，再明确运行 17b 的两回合探测。已完成的 14b/14c 不需重跑。
新会话先按顶部路线恢复到 14a 接口检查，再跳到这里。

每个设置/条件/seed 使用独立目录；新的 50 步对照也记录同样的计时协议，便于公平比较耗时。
使用 --policy.n_action_steps 加载策略，随后核对真实配置和队列容量；不在模型加载后只改一个 config 属性。
每对核对实际首噪声、初始物理状态和实际语言；后续重规划频率不同，不能声称所有后续噪声和状态相同。
计时包括 _get_action_chunk 内准备与 GPU 同步执行，不含权重加载；同时保存包含加载的总耗时。
这仍是同步仿真评测，不宣称达到真机实时控制要求。


In [ ]:
# 17a：只定义入口，不加载模型、不启动评测。
REPLAN_ROOT = Path("/content/outputs/smolvla_replan_20261031_001")
REPLAN_ARCHIVE = Path("/content/drive/MyDrive/robot-lab/experiments") / REPLAN_ROOT.name


def run_replanning_pair(condition, seed, *, action_steps=(50, 10), run_root=None, archive_root=None):
    assert GRASP_INTERFACES_VERIFIED, "请先通过 14a 真实接口检查"
    assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Drive"
    settings = tuple(action_steps)
    if len(settings) != 2 or len(set(settings)) != 2 or any(type(x) is not int or not 1 <= x <= 50 for x in settings):
        raise ValueError("需要两个不同的、1–50 范围内的整数执行窗口")
    root = Path(run_root) if run_root is not None else REPLAN_ROOT
    archive = Path(archive_root) if archive_root is not None else REPLAN_ARCHIVE
    results = {}
    for steps in settings:
        results[steps] = run_controlled_camera_pilot(
            condition, seed=seed, capture_grasp=True,
            run_root=root, archive_root=archive, n_action_steps=steps,
        )
    directories = [root / f"n_action_steps_{steps}" / condition / f"seed_{seed}" for steps in settings]
    states = [torch.load(path / "initial_state.pt", map_location="cpu", weights_only=True) for path in directories]
    noises = [torch.load(path / "first_noise.pt", map_location="cpu", weights_only=True) for path in directories]
    languages = [json.loads((path / "policy_language_audit.json").read_text()) for path in directories]
    cameras = [json.loads((path / "initial_audit.json").read_text()) for path in directories]
    checks = {
        "initial_state_equal": states[0].keys() == states[1].keys() and all(torch.equal(states[0][key], states[1][key]) for key in states[0]),
        "first_noise_equal": torch.equal(noises[0], noises[1]),
        "language_tokens_equal": languages[0]["token_ids"] == languages[1]["token_ids"] and languages[0]["attention_mask"] == languages[1]["attention_mask"],
        "camera_equal": all(cameras[0][key] == cameras[1][key] for key in ["agentview_world_position", "agentview_world_rotation", "agentview_fovy"]),
    }
    # 默认 50/10 的旧审计文件保持原格式；新的窗口组合明确记在文件名与实际配置核对中。
    suffix = "" if settings == (50, 10) else f"_{settings[0]}_vs_{settings[1]}"
    if settings != (50, 10):
        configs = [json.loads((path / "actual_policy_config.json").read_text()) for path in directories]
        checks["actual_settings_match"] = configs == [
            {"n_action_steps": steps, "chunk_size": 50, "num_steps": 10} for steps in settings]
    pair_dir = root / "pair_checks"
    pair_dir.mkdir(parents=True, exist_ok=True)
    (pair_dir / f"{condition}_seed_{seed}{suffix}.json").write_text(json.dumps(checks, indent=2))
    archive_camera_run(pair_dir, archive / "pair_checks")
    if not all(checks.values()):
        raise RuntimeError(f"配对条件未通过，保留数据并先排查：{checks}")
    for steps, result in results.items():
        timing = json.loads((root / f"n_action_steps_{steps}" / condition / f"seed_{seed}" / "timing.json").read_text())
        print({"n_action_steps": steps, "success_percent": result["overall"]["pc_success"],
               "chunk_calls": timing["chunk_calls"], "chunk_wall_s_total": timing["chunk_wall_s_total"],
               "mean_chunk_wall_s": timing["chunk_wall_s_mean"]})
    return results


In [ ]:
# 17b：明确准备好运行 GPU 后，取消下面一行注释并执行，先只做 view_15 / seed=1000。
replan_probe = run_replanning_pair("view_15", 1000)


In [ ]:
# 17c：探测回合的配置、数据和归档确认后，再扩大到原来的三个种子与默认相机对照。
# 首个探测已完成的回合会核对配置后复用，总共最多 12 个新回合（含探测的 2 个）。
for condition in ["view_15", "default_camera"]:
    for seed in [1000, 1001, 1002]:
        run_replanning_pair(condition, seed)


In [ ]:
# 17d：GPU 对照完成后，运行本单元的定义，再明确调用汇总；需要先完成 first_true_time()。
def summarize_replanning_experiment():
    records = []
    for run_dir in sorted(REPLAN_ROOT.glob("n_action_steps_*/*/seed_*")):
        if not (run_dir / "eval_info.json").is_file():
            continue  # 中断回合不当作完整回合统计
        row = summarize_grasp_run(run_dir)
        pair_file = REPLAN_ROOT / "pair_checks" / f"{row['condition']}_seed_{row['seed']}.json"
        if not pair_file.is_file() or not all(json.loads(pair_file.read_text()).values()):
            raise RuntimeError(f"该回合的 50/10 配对尚未验证通过：{pair_file}")
        timing = json.loads((run_dir / "timing.json").read_text())
        row.update({"chunk_calls": timing["chunk_calls"],
                    "chunk_wall_s_total": timing["chunk_wall_s_total"],
                    "chunk_wall_s_mean": timing["chunk_wall_s_mean"],
                    "env_steps": timing["env_steps"]})
        records.append(row)
    if not records:
        raise FileNotFoundError("尚无完整对照回合；先运行 17b")
    save_grasp_summary(records, REPLAN_ROOT / "analysis/replanning_summary.csv")
    archive_camera_run(REPLAN_ROOT / "analysis", REPLAN_ARCHIVE / "analysis")
    return records

replanning_rows = summarize_replanning_experiment()


## 18．基于实测计算预算的唯一追加候选：20 步

50/10 对照已完成：view_15 为 1/3 → 3/3，默认均为 3/3。10 步组单块平均约 0.596 秒，
95 分位约 0.631 秒，全部 90 次调用超过 0.5 秒执行窗口。保持权重、chunk_size=50、num_steps=10，
只追加 n_action_steps=20（1 秒仿真窗口），先检查 view_15 / seed=1000、1002 两个原失败样本。

这里是吞吐预算参照：即便单块生成小于 1 秒，当前 select_action 仍是队列空后同步生成，
不能据此宣称已经实现连续 20 Hz 控制；真正部署还需异步预取及端到端时延验证。

**现有会话：**直接执行下面定义单元，随后手动启用两次调用。复用原 50 步基线，每个 seed 只新增一个回合。
新会话需先恢复依赖/资源、参考审计与模型定义，通过 14a；还需执行 first_true_time 与 summarize_grasp_run 的定义。
原 50 步数据若不在 /content，会从已挂载的 Drive 归档读取；不要求重新运行原 12 回合。

决策预先固定：若两例都成功，再补 view_15 /1001 与默认相机三例，共最多新增六回合；
若未改善，不继续密集扫描窗口，保留负结果并记录 10 步方案只在当前同步仿真中有效。
随后冻结候选，用尚未参与选择的 view_5 / view_10 做验证；当前仍不能声称跨任务或跨初始化泛化。


In [ ]:
# 18a：只定义；复用现有 50 步基线，不自动启动 GPU。
BUDGET_ROOT = Path("/content/outputs/smolvla_replan_budget_20261031_001")
BUDGET_ARCHIVE = Path("/content/drive/MyDrive/robot-lab/experiments") / BUDGET_ROOT.name


def run_budget_probe(seed, condition="view_15"):
    assert GRASP_INTERFACES_VERIFIED, "请先通过 14a 接口检查"
    assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Drive"
    baseline = REPLAN_ROOT / "n_action_steps_50" / condition / f"seed_{seed}"
    if not (baseline / "eval_info.json").is_file():
        baseline = REPLAN_ARCHIVE / "n_action_steps_50" / condition / f"seed_{seed}"
    assert (baseline / "timing.json").is_file(), "未找到上一轮已完成的 50 步基线，请核对归档路径"
    result = run_controlled_camera_pilot(
        condition, seed=seed, capture_grasp=True, n_action_steps=20,
        run_root=BUDGET_ROOT, archive_root=BUDGET_ARCHIVE,
    )
    candidate = BUDGET_ROOT / "n_action_steps_20" / condition / f"seed_{seed}"
    manifests = [json.loads((path / "manifest.json").read_text()) for path in [baseline, candidate]]
    initial = [torch.load(path / "initial_state.pt", map_location="cpu", weights_only=True) for path in [baseline, candidate]]
    noise = [torch.load(path / "first_noise.pt", map_location="cpu", weights_only=True) for path in [baseline, candidate]]
    language = [json.loads((path / "policy_language_audit.json").read_text()) for path in [baseline, candidate]]
    audits = [json.loads((path / "initial_audit.json").read_text()) for path in [baseline, candidate]]
    configs = [json.loads((path / "actual_policy_config.json").read_text()) for path in [baseline, candidate]]
    checks = {
        "checkpoint_task_seed_equal": all(manifests[0][key] == manifests[1][key] for key in [
            "condition", "seed", "checkpoint", "checkpoint_revision", "libero_plus_revision", "asset_revision", "task_name"]),
        "actual_settings_correct": configs == [
            {"n_action_steps": 50, "chunk_size": 50, "num_steps": 10},
            {"n_action_steps": 20, "chunk_size": 50, "num_steps": 10}],
        "initial_state_equal": initial[0].keys() == initial[1].keys() and all(torch.equal(initial[0][key], initial[1][key]) for key in initial[0]),
        "first_noise_equal": torch.equal(noise[0], noise[1]),
        "language_equal": all(language[0][key] == language[1][key] for key in ["token_ids", "attention_mask"]),
        "camera_control_equal": all(audits[0][key] == audits[1][key] for key in [
            "agentview_world_position", "agentview_world_rotation", "agentview_fovy", "control_mode", "control_frequency", "settling_steps"]),
    }
    pair_dir = BUDGET_ROOT / "pair_checks"
    pair_dir.mkdir(parents=True, exist_ok=True)
    (pair_dir / f"{condition}_seed_{seed}_50_vs_20.json").write_text(json.dumps({
        "baseline_source": str(baseline), "candidate_source": str(candidate), "checks": checks,
    }, ensure_ascii=False, indent=2))
    archive_camera_run(pair_dir, BUDGET_ARCHIVE / "pair_checks")
    if not all(checks.values()):
        raise RuntimeError(f"20 步探测的配对未通过，保留结果并先排查：{checks}")
    rows = []
    for directory in [baseline, candidate]:
        row = summarize_grasp_run(directory)
        timing = json.loads((directory / "timing.json").read_text())
        row.update({"chunk_calls": timing["chunk_calls"], "chunk_wall_s_total": timing["chunk_wall_s_total"],
                    "chunk_wall_s_mean": timing["chunk_wall_s_mean"], "env_steps": timing["env_steps"]})
        rows.append(row)
    save_grasp_summary(rows, BUDGET_ROOT / "analysis" / f"{condition}_seed_{seed}_50_vs_20.csv")
    archive_camera_run(BUDGET_ROOT / "analysis", BUDGET_ARCHIVE / "analysis")
    return result


In [ ]:
# 18b：先只新增两个回合。准备运行 GPU 时取消注释。
budget_seed1000 = run_budget_probe(1000)
budget_seed1002 = run_budget_probe(1002)


In [ ]:
# 18c：只有前两例都成功，才补剩余四回合；否则停止窗口扫描，记录负结果。
run_budget_probe(1001)
for seed in [1000, 1001, 1002]:
    run_budget_probe(seed, condition="default_camera")


## 19．冻结 20 步候选，做未参与选择的视角验证

20 步候选已完成六回合：默认相机和 view_15 均 3/3，view_15 每回合模型生成总耗时约 4.14 s，
相较 10 步的约 9.33 s 减少约 55.6%。至此停止选择窗口，固定比较 50 与 20。

保留验证条件已预先确认：view_5（ID 656）、view_10（ID 661），均是同一基础任务和 initstate_0 后缀。
每个视角三个种子，两个设置，共 12 个新回合。它们验证未参与选参数的相机条件，仍不代表跨任务或跨初始化。
探测回合只用于检查接口/归档；不依据它的成功与失败再改参数。

**现有会话按此顺序：**
1. 重新执行第 11 节含 controlled_camera_task / run_controlled_camera_pilot 的定义单元（加入两种任务映射）。
2. 重新执行 17a 的定义单元（接口支持指定 50/20，旧 50/10 调用保持兼容）。不要重跑旧实验调用。
3. 执行 19a 定义，然后手动启用 19b 的首对。
4. 只要接口和配对有效，无论首对是否成功，都按预先固定条件执行 19c，最后运行 19d 汇总。

协议在首次调用前落盘并归档；已有协议不同时要求新目录，不悄悄改变验证集合。
全部新 GPU 调用默认注释。验证完成后先写结论，不继续根据这批保留条件重选窗口。


In [ ]:
# 19a：只定义，不启动 GPU。
import inspect
HELDOUT_ROOT = Path("/content/outputs/smolvla_heldout_camera_20261031_001")
HELDOUT_ARCHIVE = Path("/content/drive/MyDrive/robot-lab/experiments") / HELDOUT_ROOT.name
FROZEN_HELDOUT_TASKS = {"view_5": 656, "view_10": 661}
FROZEN_HELDOUT_SEEDS = (1000, 1001, 1002)


def freeze_heldout_protocol():
    for condition, task_id in FROZEN_HELDOUT_TASKS.items():
        expected = f"{pilot_base_task}_{condition}_15_100_0_0_initstate_0"
        if benchmark.libero_task_map["libero_object"][task_id] != expected:
            raise RuntimeError(f"保留任务名称不符，停止：{condition}, {task_id}")
    protocol = {
        "purpose": "camera validation after selecting on default/view_15; no retuning on heldout results",
        "candidate_n_action_steps": 20, "reference_n_action_steps": 50,
        "chunk_size": 50, "num_steps": 10,
        "conditions": FROZEN_HELDOUT_TASKS, "seeds": list(FROZEN_HELDOUT_SEEDS),
        "initial_state_index": 0, "control_frequency_hz": 20, "control_mode": "relative",
        "checkpoint": CHECKPOINT, "checkpoint_revision": checkpoint_revision,
        "libero_plus_revision": PLUS_SHA, "asset_revision": ASSET_REVISION,
        "language": pilot_instruction,
        "scope": "same task and initialization; unseen camera variants, not cross-task or cross-initialization",
    }
    protocol_dir = HELDOUT_ROOT / "protocol"
    protocol_dir.mkdir(parents=True, exist_ok=True)
    protocol_file = protocol_dir / "heldout_protocol.json"
    if protocol_file.exists() and json.loads(protocol_file.read_text()) != protocol:
        raise RuntimeError("已有保留验证协议不同；请使用新目录，并说明为何变更")
    protocol_file.write_text(json.dumps(protocol, ensure_ascii=False, indent=2))
    archive_camera_run(protocol_dir, HELDOUT_ARCHIVE / "protocol")
    return protocol


def run_heldout_pair(condition, seed):
    assert Path("/content/drive/MyDrive").is_dir(), "先挂载 Drive"
    if condition not in FROZEN_HELDOUT_TASKS or seed not in FROZEN_HELDOUT_SEEDS:
        raise ValueError("请使用预先固定的保留条件和种子")
    if "action_steps" not in inspect.signature(run_replanning_pair).parameters:
        raise RuntimeError("内核仍是旧定义，请重新执行 17a 定义单元")
    try:
        controlled_camera_task(condition)
    except KeyError as error:
        raise RuntimeError("请重新执行第 11 节的评测定义单元，加载新的任务映射") from error
    freeze_heldout_protocol()
    return run_replanning_pair(
        condition, seed, action_steps=(50, 20), run_root=HELDOUT_ROOT, archive_root=HELDOUT_ARCHIVE,
    )


In [ ]:
# 19b：先检查一对的接口和配对；不是根据结果选择参数。
heldout_probe = run_heldout_pair("view_10", 1000)


In [ ]:
# 19c：首对接口与归档有效后执行固定集合，共 12 个新回合（含首对，完成结果自动复用）。
for condition in FROZEN_HELDOUT_TASKS:
    for seed in FROZEN_HELDOUT_SEEDS:
        run_heldout_pair(condition, seed)


In [ ]:
# 19d：仅定义汇总。可在首对后看进度，也可在全部完成后生成最终表。
def summarize_heldout_validation():
    freeze_heldout_protocol()
    rows = []
    for condition in FROZEN_HELDOUT_TASKS:
        for seed in FROZEN_HELDOUT_SEEDS:
            directories = [HELDOUT_ROOT / f"n_action_steps_{steps}" / condition / f"seed_{seed}" for steps in (50, 20)]
            if not all((directory / "eval_info.json").is_file() for directory in directories):
                continue
            audit_file = HELDOUT_ROOT / "pair_checks" / f"{condition}_seed_{seed}_50_vs_20.json"
            if not audit_file.is_file() or not all(json.loads(audit_file.read_text()).values()):
                raise RuntimeError(f"保留回合的配对未通过：{audit_file}")
            for directory in directories:
                row = summarize_grasp_run(directory)
                timing = json.loads((directory / "timing.json").read_text())
                row.update({"chunk_calls": timing["chunk_calls"],
                            "chunk_wall_s_total": timing["chunk_wall_s_total"],
                            "chunk_wall_s_mean": timing["chunk_wall_s_mean"], "env_steps": timing["env_steps"]})
                rows.append(row)
    if not rows:
        raise FileNotFoundError("尚无完整且核对通过的保留配对")
    completed_pairs = len(rows) // 2
    expected_pairs = len(FROZEN_HELDOUT_TASKS) * len(FROZEN_HELDOUT_SEEDS)
    # 进度与最终表用不同文件名，归档时不覆盖已保存的快照。
    save_grasp_summary(rows, HELDOUT_ROOT / "analysis" / f"heldout_summary_{completed_pairs}pairs.csv")
    groups = []
    for condition in FROZEN_HELDOUT_TASKS:
        for steps in (50, 20):
            group = [row for row in rows if row["condition"] == condition and row["n_action_steps"] == steps]
            if group:
                groups.append({"condition": condition, "n_action_steps": steps,
                               "successes": sum(row["success"] for row in group), "episodes": len(group),
                               "mean_chunk_wall_s_total": sum(row["chunk_wall_s_total"] for row in group) / len(group)})
    report = {"completed_pairs": completed_pairs, "expected_pairs": expected_pairs,
              "validation_complete": completed_pairs == expected_pairs, "groups": groups}
    (HELDOUT_ROOT / "analysis" / f"heldout_report_{completed_pairs}pairs.json").write_text(json.dumps(report, indent=2))
    archive_camera_run(HELDOUT_ROOT / "analysis", HELDOUT_ARCHIVE / "analysis")
    print("已完成配对：", completed_pairs, "/", expected_pairs)
    for group in groups:
        print(group)
    return report

heldout_report = summarize_heldout_validation()
